In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import rankdata, spearmanr, wilcoxon, friedmanchisquare
from sklearn.ensemble import RandomForestRegressor

import scikit_posthocs as sp

warnings.filterwarnings("ignore")
np.random.seed(42)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SOURCE_DIR = os.path.join(PROJECT_ROOT, "source")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

ALGOS = ["DT", "RF", "SVM", "NB", "KNN"]
K = len(ALGOS)          # numero de algoritmos
ALPHA = 0.05            # nivel de significancia do teste de Wilcoxon
SEED = 42

plt.rcParams["figure.dpi"] = 110



## 1. O meta-dataset

### 1.1 A matriz de performances P

Carregamos $P$ já pronta a partir de `results/MatrizP_30.csv`, gerada pelo
notebook `01_gerarMatriz_P_e_X.ipynb`. Cada célula de $P$ é a **acurácia
média** (sobre 10 repetições de hold-out estratificado 80/20) de um
algoritmo em um dataset — a métrica e a forma de obtenção estão descritas e
justificadas naquele notebook (métrica escolhida: **acurácia**, por se
tratar de uma coleção inteiramente de classificação).

Também carregamos o detalhamento **por fold**
(`MatrizP_folds_30.csv`), necessário mais adiante para o teste de Wilcoxon
pareado (vitórias significativas, Seção 2.1.3).


In [ ]:

P_path = os.path.join(RESULTS_DIR, "MatrizP_30.csv")
folds_path = os.path.join(RESULTS_DIR, "MatrizP_folds_30.csv")

assert os.path.exists(P_path) and os.path.exists(folds_path), (
    "Rode antes o notebook 01_gerarMatriz_P_e_X.ipynb para gerar "
    "results/MatrizP_30.csv e results/MatrizP_folds_30.csv."
)

P_df = pd.read_csv(P_path, index_col="Dataset")
folds_df = pd.read_csv(folds_path)

dataset_names = list(P_df.index)   # ordem alfabetica
P = P_df[ALGOS].copy()             # matriz P "pura" (sem a coluna Best), 30 x 5

print(f"Datasets: {len(dataset_names)}   |   Algoritmos: {ALGOS}")
P.head(10)

Datasets: 30   |   Algoritmos: ['DT', 'RF', 'SVM', 'NB', 'KNN']


,DT,RF,SVM,NB,KNN
Dataset,,,,,
annealing,0.953889,0.958333,0.852778,0.538889,0.878889
balloons,0.600000,0.625000,0.650000,0.525000,0.650000
bank,0.863094,0.900221,0.894144,0.816796,0.892486
blood,0.719333,0.739333,0.762667,0.734667,0.752667
car,0.980347,0.984393,0.968497,0.691908,0.935549
contrac,0.495593,0.525763,0.549492,0.467119,0.478644
dermatology,0.956757,0.974324,0.979730,0.870270,0.971622
echocardiogram,0.744444,0.844444,0.837037,0.874074,0.814815
ecoli,0.795588,0.894118,0.888235,0.780882,0.873529



## Meta-características X

Detalhameto das meta-características extraídas:

| Família (enunciado) | Família (`pymfe`) | Nº | Exemplos |
|:--|:--|--:|:--|
| Gerais / de dimensão | `general` | 11 | `nr_inst`, `nr_attr`, `nr_class`, `nr_num`, `nr_cat`, `nr_bin`, `attr_to_inst`, `inst_to_attr`, `cat_to_num` |
| Balanceamento de classes | `general` | (inclusas acima) | `freq_class.mean`, `freq_class.sd` |
| Estatísticas | `statistical` | 42 | `mean.*`, `sd.*`, `var.*`, `median.*`, `min.*`, `max.*`, `range.*`, `iq_range.*`, `mad.*`, `skewness.*`, `kurtosis.*`, `cor.*`, `cov.*`, `eigenvalues.*`, `nr_outliers`, `nr_norm`, `w_lambda` |
| Teoria da informação | `info-theory` | 13 | `class_ent`, `attr_ent.*`, `joint_ent.*`, `mut_inf.*`, `ns_ratio`, `eq_num_attr`, `attr_conc.*`, `class_conc.*` |
| Derivadas | `general` / `model-based` | (inclusas) | `attr_to_inst`, `inst_to_attr`, `cat_to_num`, `nodes_per_attr`, `nodes_per_inst` — razões calculadas internamente pelo `pymfe` |
| — (extras, sem correspondente no enunciado) | `model-based` | 24 | `leaves*`, `nodes*`, `tree_depth.*`, `tree_imbalance.*`, `var_importance.*` |
| — | `complexity` | 23 | `c1`, `c2`, `f1.*`, `f3`, `f4`, `l1`, `l2`, `l3`, `n1`, `n3.*`, `n4.*`, `t1`–`t4`, `density`, `lsc` |
| — | `clustering` | 8 | `ch`, `sil`, `vdb`, `vdu`, `nre`, `pb`, `sc`, `int` |
| — | `concept` | 8 | `cohesiveness.*`, `conceptvar.*`, `impconceptvar.*`, `wg_dist.*` |
| — | `itemset` | 4 | `one_itemset.*`, `two_itemset.*` |
| **Total** | | **133** | |

In [ ]:

X_full = pd.read_csv(os.path.join(SOURCE_DIR, "MatrizX.csv"), index_col="Dataset")
X = X_full.loc[dataset_names].copy()

all_nan_cols = X.columns[X.isna().all()] # descarta colunas 1com NaN
X = X.drop(columns=all_nan_cols)
print(f"Colunas descartadas (100% NaN): {list(all_nan_cols)}")

# imputa os poucos NaN com a media da coluna
remaining_na = X.isna().sum()
remaining_na = remaining_na[remaining_na > 0]
if len(remaining_na):
    print(f"Colunas com poucos NaN (imputados pela média): {list(remaining_na.index)}")
X = X.fillna(X.mean())

assert not X.isna().any().any(), "Ainda há NaN em X depois da imputação!"
print(f"\nX final: {X.shape[0]} datasets x {X.shape[1]} meta-características")
X.head(5)


## Matriz de ranks R

Convertendo $P$ em ranks: como a métrica é acurácia (**maior é melhor**), fiz o ranking de cada linha em ordem decrescente de pformance.
Empates recebem o rank médio (`scipy.stats.rankdata`,
`method="average"`).


In [4]:

def ranks_from_performance(P_row_or_df, higher_is_better=True):
    '''
    Converte performances em ranks (1 = melhor), linha a linha.
    Aceita tanto uma linha (array 1D) quanto uma matriz completa (2D).
    '''
    arr = np.asarray(P_row_or_df, dtype=float)
    sign = -1.0 if higher_is_better else 1.0
    if arr.ndim == 1:
        return rankdata(sign * arr, method="average")
    return np.vstack([rankdata(sign * row, method="average") for row in arr])


R = pd.DataFrame(
    ranks_from_performance(P.values, higher_is_better=True),
    index=P.index, columns=P.columns,
)
R.head(10)


,DT,RF,SVM,NB,KNN
Dataset,,,,,
annealing,2.0,1.0,4.0,5.0,3.0
balloons,4.0,3.0,1.5,5.0,1.5
bank,4.0,1.0,2.0,5.0,3.0
blood,5.0,3.0,1.0,4.0,2.0
car,2.0,1.0,3.0,5.0,4.0
contrac,3.0,2.0,1.0,5.0,4.0
dermatology,4.0,2.0,1.0,5.0,3.0
echocardiogram,5.0,2.0,3.0,1.0,4.0
ecoli,4.0,1.0,2.0,5.0,3.0


## As 6 abordagens


### 1. AR — rank médio

In [5]:

def aggregate_rank_mean(R_subset):
    '''
    AR - rank medio: media dos ranks por algoritmo (coluna), reranqueada.
    R_subset: array (n_datasets, K).
    '''
    mean_ranks = np.asarray(R_subset, dtype=float).mean(axis=0)
    return rankdata(mean_ranks, method="average")


AR_global = aggregate_rank_mean(R.values)
print("Ranking AR (rank médio), todos os 30 datasets:")
for algo, r in sorted(zip(ALGOS, AR_global), key=lambda t: t[1]):
    print(f"  {algo:4s} -> rank {r:.2f}")


Ranking AR (rank médio), todos os 30 datasets:
  RF   -> rank 1.00
  SVM  -> rank 2.00
  KNN  -> rank 3.00
  DT   -> rank 4.00
  NB   -> rank 5.00


### 2. MR — rank mediano

In [6]:

def aggregate_rank_median(R_subset):
    '''MR - rank mediano: mediana dos ranks por algoritmo, reranqueada.'''
    median_ranks = np.median(np.asarray(R_subset, dtype=float), axis=0)
    return rankdata(median_ranks, method="average")


MR_global = aggregate_rank_median(R.values)
print("Ranking MR (rank mediano), todos os 30 datasets:")
for algo, r in sorted(zip(ALGOS, MR_global), key=lambda t: t[1]):
    print(f"  {algo:4s} -> rank {r:.2f}")


Ranking MR (rank mediano), todos os 30 datasets:
  RF   -> rank 1.50
  SVM  -> rank 1.50
  KNN  -> rank 3.00
  DT   -> rank 4.00
  NB   -> rank 5.00


### 3. Vitórias significativas

In [ ]:

from itertools import combinations

# fold_matrix[dataset][algo] = array com os 10 valores de acuracia por fold
fold_matrix = {
    name: {
        algo: folds_df.query("dataset == @name and algoritmo == @algo")
                       .sort_values("fold")["accuracy"].values
        for algo in ALGOS
    }
    for name in dataset_names
}

pairs = list(combinations(range(K), 2))  # pares de indices de algoritmos (i<j)

# win_table[d, p] = +1 se ALGOS[pairs[p][0]] venceu de forma significativa,
#                   -1 se ALGOS[pairs[p][1]] venceu,
#                    0 se nao houve diferenca significativa
win_table = np.zeros((len(dataset_names), len(pairs)), dtype=int)

for d_idx, name in enumerate(dataset_names):
    for p_idx, (i, j) in enumerate(pairs):
        a, b = fold_matrix[name][ALGOS[i]], fold_matrix[name][ALGOS[j]]
        if np.allclose(a, b):
            continue  # Wilcoxon exige diferencas nao todas nulas
        try:
            _, p_value = wilcoxon(a, b)
        except ValueError:
            continue  # todas as diferencas zero (empate perfeito)
        if p_value < ALPHA:
            win_table[d_idx, p_idx] = 1 if a.mean() > b.mean() else -1

print(f"Testes de Wilcoxon realizados: {len(dataset_names) * len(pairs)} "
      f"({len(dataset_names)} datasets x {len(pairs)} pares)")
print(f"Proporção de comparações significativas: "
      f"{(win_table != 0).mean():.1%}")


def aggregate_significant_wins(dataset_idx_subset):
    '''
    Conta, para cada algoritmo, o total de vitorias significativas somando apenas as linhas de win_table associadas aos datasets em dataset_idx_subset.
    Retorna o ranking (rank 1 = mais vitorias).
    '''
    wins = np.zeros(K)
    sub = win_table[dataset_idx_subset]
    for p_idx, (i, j) in enumerate(pairs):
        wins[i] += (sub[:, p_idx] == 1).sum()
        wins[j] += (sub[:, p_idx] == -1).sum()
    return rankdata(-wins, method="average"), wins


WINS_global, wins_count_global = aggregate_significant_wins(np.arange(len(dataset_names)))
print("\nVitórias significativas totais (30 datasets) e ranking resultante:")
for algo, wins, r in sorted(zip(ALGOS, wins_count_global, WINS_global), key=lambda t: t[2]):
    print(f"  {algo:4s} -> {int(wins):3d} vitórias -> rank {r:.2f}")


Testes de Wilcoxon realizados: 300 (30 datasets x 10 pares)
Proporção de comparações significativas: 59.3%

Vitórias significativas totais (30 datasets) e ranking resultante:
  RF   ->  60 vitórias -> rank 1.00
  SVM  ->  55 vitórias -> rank 2.00
  KNN  ->  30 vitórias -> rank 3.00
  DT   ->  23 vitórias -> rank 4.00
  NB   ->  10 vitórias -> rank 5.00


### 4. Abordagem 1 — regressor de múltiplas saídas sobre P

In [ ]:
def fit_predict_abordagem1(X_train, P_train, x_test, seed=SEED):
    '''
    Abordagem 1: regressor multi-saida treinado sobre P (performances).
    Retorna o ranking predito para x_test (1 = melhor).
    '''
    model = RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=1)
    model.fit(X_train, P_train)
    pred_perf = model.predict(x_test.reshape(1, -1))[0]
    return rankdata(-pred_perf, method="average"), pred_perf


rank_pred, perf_pred = fit_predict_abordagem1(
    X.values, P.values, X.values[0])
print(f"Exemplo (treinando com os 30, predizendo para '{dataset_names[0]}'):")
print("Performance predita:", dict(zip(ALGOS, np.round(perf_pred, 3))))
print("Ranking predito     :", dict(zip(ALGOS, rank_pred)))
print("Ranking real (R)    :", dict(zip(ALGOS, R.iloc[0].values)))

Exemplo (treinando com os 30, predizendo para 'annealing'):
Performance predita: {'DT': np.float64(0.928), 'RF': np.float64(0.944), 'SVM': np.float64(0.872), 'NB': np.float64(0.624), 'KNN': np.float64(0.882)}
Ranking predito     : {'DT': np.float64(2.0), 'RF': np.float64(1.0), 'SVM': np.float64(4.0), 'NB': np.float64(5.0), 'KNN': np.float64(3.0)}
Ranking real (R)    : {'DT': np.float64(2.0), 'RF': np.float64(1.0), 'SVM': np.float64(4.0), 'NB': np.float64(5.0), 'KNN': np.float64(3.0)}


#### 5. Abordagem 2 — regressor de múltiplas saídas sobre R

In [ ]:
def fit_predict_abordagem2(X_train, R_train, x_test, seed=SEED):
    '''
    Abordagem 2: regressor multi-saida treinado sobre R (ranks), prediz
    as posicoes diretamente. Retorna o ranking predito (1 = melhor).
    '''
    model = RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1)
    model.fit(X_train, R_train)
    pred_rank_raw = model.predict(x_test.reshape(1, -1))[0]
    return rankdata(pred_rank_raw, method="average"), pred_rank_raw

rank_pred2, raw_pred2 = fit_predict_abordagem2(
    X.values, R.values, X.values[0])
print(f"Exemplo (treinando com os 30, predizendo para '{dataset_names[0]}'):")
print("Rank bruto predito :", dict(zip(ALGOS, np.round(raw_pred2, 2))))
print("Ranking predito    :", dict(zip(ALGOS, rank_pred2)))
print("Ranking real (R)   :", dict(zip(ALGOS, R.iloc[0].values)))


Exemplo (treinando com os 30, predizendo para 'annealing'):
Rank bruto predito : {'DT': np.float64(2.58), 'RF': np.float64(1.14), 'SVM': np.float64(3.35), 'NB': np.float64(4.82), 'KNN': np.float64(3.12)}
Ranking predito    : {'DT': np.float64(2.0), 'RF': np.float64(1.0), 'SVM': np.float64(4.0), 'NB': np.float64(5.0), 'KNN': np.float64(3.0)}
Ranking real (R)   : {'DT': np.float64(2.0), 'RF': np.float64(1.0), 'SVM': np.float64(4.0), 'NB': np.float64(5.0), 'KNN': np.float64(3.0)}


### 6. HARRIS


In [ ]:

def scale_cols_01(P_mat):
    '''Escala cada coluna de P para [0,1] (min-max), evitando divisao por 0.'''
    pmin = P_mat.min(axis=0)
    pmax = P_mat.max(axis=0)
    span = np.where(pmax - pmin == 0, 1.0, pmax - pmin)
    return (P_mat - pmin) / span


def borda_consensus_rank(R_subset):
    return rankdata(R_subset.mean(axis=0), method="average")


def harris_regression_loss(Ps_subset, y_reg):
    '''MSE medio (por instancia e por algoritmo) em relacao ao rotulo do nó.'''
    if len(Ps_subset) == 0:
        return 0.0
    return float(np.mean((Ps_subset - y_reg) ** 2))


def harris_ranking_loss(R_subset, y_rank):
    '''
    Spearman(rank da instancia, ranking de consenso do no), em [0,1]. 
    '''
    # Como R_subset e y_rank ja sao ranks, o coeficiente de Spearman e apenas a correlacao de Pearson entre eles calculada de forma vetorizada
    # (uma linha por instancia) por eficiencia (chamar scipy.stats.spearmanr instancia a instancia seria ordens de magnitude mais lento aqui).
    
    if len(R_subset) == 0:
        return 0.0
    Rc = R_subset - R_subset.mean(axis=1, keepdims=True)
    yc = y_rank - y_rank.mean()
    num = (Rc * yc).sum(axis=1)
    den = np.sqrt((Rc ** 2).sum(axis=1) * (yc ** 2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        rho = np.where(den > 1e-12, num / den, 0.0)
    return float(np.mean((1.0 - rho) / 2.0))


def harris_node_loss(Ps_subset, R_subset, lam):
    '''Perda combinada L(D) = lambda * L_ranking + (1 - lambda) * L_regressao.'''
    y_reg = Ps_subset.mean(axis=0)
    y_rank = borda_consensus_rank(R_subset)
    Lreg = harris_regression_loss(Ps_subset, y_reg)
    Lrank = harris_ranking_loss(R_subset, y_rank)
    return lam * Lrank + (1 - lam) * Lreg

In [ ]:

class _HarrisNode:
    __slots__ = ("feature", "threshold", "left", "right", "y_reg", "is_leaf")


class HarrisTree:
    def __init__(self, lam=0.5, max_depth=3, min_samples_leaf=3,
                 max_features=None, random_state=None):
        self.lam = lam
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.rng = np.random.RandomState(random_state)

    def fit(self, X_mat, Ps_scaled, R_mat):
        idx = np.arange(len(X_mat))
        self.root = self._build(X_mat, Ps_scaled, R_mat, idx, depth=0)
        return self

    def _make_leaf(self, Ps_scaled, idx):
        node = _HarrisNode()
        node.is_leaf = True
        node.y_reg = Ps_scaled[idx].mean(axis=0)
        return node

    def _build(self, X_mat, Ps_scaled, R_mat, idx, depth):
        if depth >= self.max_depth or len(idx) < 2 * self.min_samples_leaf:
            return self._make_leaf(Ps_scaled, idx)

        n_features = X_mat.shape[1]
        if self.max_features is None or self.max_features >= n_features:
            feat_candidates = np.arange(n_features)
        else:
            feat_candidates = self.rng.choice(
                n_features, size=self.max_features, replace=False)

        best = None  # (perda_ponderada, feature, limiar, idx_esq, idx_dir)
        for f in feat_candidates:
            vals = X_mat[idx, f]
            uniq = np.unique(vals)
            if len(uniq) < 2:
                continue
            thresholds = (uniq[:-1] + uniq[1:]) / 2.0
            for thr in thresholds:
                left_mask = vals <= thr
                n_left = int(left_mask.sum())
                n_right = len(idx) - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue
                left_idx = idx[left_mask]
                right_idx = idx[~left_mask]
                Ll = harris_node_loss(Ps_scaled[left_idx], R_mat[left_idx], self.lam)
                Lr = harris_node_loss(Ps_scaled[right_idx], R_mat[right_idx], self.lam)
                weighted = (n_left * Ll + n_right * Lr) / len(idx)
                if best is None or weighted < best[0]:
                    best = (weighted, f, thr, left_idx, right_idx)

        if best is None:
            return self._make_leaf(Ps_scaled, idx)

        _, f, thr, left_idx, right_idx = best
        node = _HarrisNode()
        node.is_leaf = False
        node.feature, node.threshold = f, thr
        node.left = self._build(X_mat, Ps_scaled, R_mat, left_idx, depth + 1)
        node.right = self._build(X_mat, Ps_scaled, R_mat, right_idx, depth + 1)
        return node

    def predict_one(self, x):
        node = self.root
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.y_reg


class HarrisForest:
    def __init__(self, n_estimators=60, lam=0.5, max_depth=3,
                 min_samples_leaf=3, max_features="sqrt", random_state=42):
        self.n_estimators = n_estimators
        self.lam = lam
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X_mat, P_mat, R_mat):
        n, n_features = X_mat.shape
        Ps_scaled = scale_cols_01(P_mat)  # escala fixa computada uma vez
        if self.max_features == "sqrt":
            mf = max(1, int(round(np.sqrt(n_features))))
        elif self.max_features is None:
            mf = n_features
        else:
            mf = self.max_features

        rng = np.random.RandomState(self.random_state)
        self.trees_ = []
        for _ in range(self.n_estimators):
            boot_idx = rng.randint(0, n, size=n)
            tree = HarrisTree(
                lam=self.lam, max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf, max_features=mf,
                random_state=rng.randint(0, 10**6),
            )
            tree.fit(X_mat[boot_idx], Ps_scaled[boot_idx], R_mat[boot_idx])
            self.trees_.append(tree)
        return self

    def predict(self, x):
        '''Media das predicoes das arvores da floresta.'''
        preds = np.array([tree.predict_one(x) for tree in self.trees_])
        return preds.mean(axis=0)


def fit_predict_harris(X_train, P_train, R_train, x_test, lam,
                        n_estimators=60, max_depth=3, min_samples_leaf=3,
                        seed=SEED):
    '''Treina uma HarrisForest e retorna o ranking predito.'''
    forest = HarrisForest(n_estimators=n_estimators, lam=lam,
                           max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                           random_state=seed)
    forest.fit(X_train, P_train, R_train)
    pred_perf_scaled = forest.predict(x_test)
    return rankdata(-pred_perf_scaled, method="average"), pred_perf_scaled

In [ ]:
# testando: treina com os 30 datasets, prediz para o primeiro.
rank_pred_harris, perf_pred_harris = fit_predict_harris(
    X.values, P.values, R.values, X.values[0], lam=0.5)
print(f"Exemplo HARRIS (lambda=0.5), predizendo para '{dataset_names[0]}':")
print("Performance predita (escalada [0,1]):",
      dict(zip(ALGOS, np.round(perf_pred_harris, 3))))
print("Ranking predito :", dict(zip(ALGOS, rank_pred_harris)))
print("Ranking real (R):", dict(zip(ALGOS, R.iloc[0].values)))


Exemplo HARRIS (lambda=0.5), predizendo para 'annealing':
Performance predita (escalada [0,1]): {'DT': np.float64(0.753), 'RF': np.float64(0.811), 'SVM': np.float64(0.751), 'NB': np.float64(0.592), 'KNN': np.float64(0.766)}
Ranking predito : {'DT': np.float64(3.0), 'RF': np.float64(1.0), 'SVM': np.float64(4.0), 'NB': np.float64(5.0), 'KNN': np.float64(2.0)}
Ranking real (R): {'DT': np.float64(2.0), 'RF': np.float64(1.0), 'SVM': np.float64(4.0), 'NB': np.float64(5.0), 'KNN': np.float64(3.0)}


## Laço leave-one-dataset-out

In [ ]:

def loss_curve_and_auc(order_best_to_worst, P_true_row):
    best_possible = P_true_row.max()
    running_best = -np.inf
    losses = []
    for algo_idx in order_best_to_worst:
        running_best = max(running_best, P_true_row[algo_idx])
        losses.append(best_possible - running_best)
    return losses, float(np.mean(losses))


def ranking_to_order(rank_vector):
    return np.argsort(rank_vector)


HARRIS_LAMBDAS = [0.0, 0.25, 0.5, 0.75, 1.0]
SIX_APPROACHES = ["AR", "MR", "Vitórias significativas", "Abordagem 1 (regressor sobre P)", "Abordagem 2 (regressor sobre R)","HARRIS (lambda=0.5)"]

In [ ]:
X_arr, P_arr, R_arr = X.values, P.values, R.values
n_datasets = len(dataset_names)
all_idx = np.arange(n_datasets)

lodo_rows = []

for test_i, test_name in enumerate(dataset_names):
    train_idx = all_idx[all_idx != test_i]
    X_train, P_train, R_train = X_arr[train_idx], P_arr[train_idx], R_arr[train_idx]
    x_test, P_true, R_true = X_arr[test_i], P_arr[test_i], R_arr[test_i]

    preds = {}
    preds["AR"] = aggregate_rank_mean(R_train)
    preds["MR"] = aggregate_rank_median(R_train)
    preds["Vitórias significativas"] = aggregate_significant_wins(train_idx)[0]
    preds["Abordagem 1 (regressor sobre P)"], _ = fit_predict_abordagem1(
        X_train, P_train, x_test, seed=SEED)
    preds["Abordagem 2 (regressor sobre R)"], _ = fit_predict_abordagem2(
        X_train, R_train, x_test, seed=SEED)
    for lam in HARRIS_LAMBDAS:
        preds[f"HARRIS (lambda={lam})"], _ = fit_predict_harris(
            X_train, P_train, R_train, x_test, lam=lam, seed=SEED + test_i)

    for approach_name, rank_pred in preds.items():
        rho, _ = spearmanr(rank_pred, R_true)
        order = ranking_to_order(rank_pred)
        losses, auc = loss_curve_and_auc(order, P_true)
        row = {"dataset": test_name, "approach": approach_name,
               "spearman": rho, "auc_loss": auc}
        row.update({f"loss_t{t+1}": losses[t] for t in range(K)})
        lodo_rows.append(row)

    print(f"[{test_i+1:2d}/{n_datasets}] LODO concluído para '{test_name}'")

lodo_df = pd.DataFrame(lodo_rows)

os.makedirs(RESULTS_DIR, exist_ok=True)
lodo_csv_path = os.path.join(RESULTS_DIR, "lodo_results_30.csv")
lodo_df.to_csv(lodo_csv_path, index=False)

print(f"\nTotal de linhas (30 datasets x {lodo_df['approach'].nunique()} configurações): {len(lodo_df)}")
print(f"Resultados brutos do LODO salvos em: {lodo_csv_path}")


[ 1/30] LODO concluído para 'annealing'
[ 2/30] LODO concluído para 'balloons'
[ 3/30] LODO concluído para 'bank'
[ 4/30] LODO concluído para 'blood'
[ 5/30] LODO concluído para 'car'
[ 6/30] LODO concluído para 'contrac'
[ 7/30] LODO concluído para 'dermatology'
[ 8/30] LODO concluído para 'echocardiogram'
[ 9/30] LODO concluído para 'ecoli'
[10/30] LODO concluído para 'fertility'
[11/30] LODO concluído para 'flags'
[12/30] LODO concluído para 'glass'
[13/30] LODO concluído para 'hepatitis'
[14/30] LODO concluído para 'ionosphere'
[15/30] LODO concluído para 'iris'
[16/30] LODO concluído para 'lenses'
[17/30] LODO concluído para 'libras'
[18/30] LODO concluído para 'lymphography'
[19/30] LODO concluído para 'mammographic'
[20/30] LODO concluído para 'nursery'
[21/30] LODO concluído para 'parkinsons'
[22/30] LODO concluído para 'pima'
[23/30] LODO concluído para 'planning'
[24/30] LODO concluído para 'seeds'
[25/30] LODO concluído para 'soybean'
[26/30] LODO concluído para 'spect'
[27/